# Anexo I · Laboratório: os padrões de candlesticks postos à prova
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**: preços ajustados das 40 ações desde 2005, com a limpeza do Capítulo 28. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_anexo1.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Os padrões são os da biblioteca **TA-Lib** (funções CDL), com as definições dela. Testes didáticos, sem custo, em ações que existem hoje (viés de sobrevivência); com 28 padrões testados, alguns parecerão bons por acaso (sobreajuste): **não são recomendação**.

In [ ]:
!pip install -q yfinance TA-Lib

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import talib                               # no Colab: !pip install TA-Lib
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
FIM = "2026-10-01"
CACHE = "dados_anexo1.pkl"
CORTE = pd.Timestamp("2016-01-01")
LUGAR = 0.05                               # "lugar certo": queda (ou alta) de 5% ou mais nos 10 pregões anteriores


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")

## Parte 1: dados e frequência dos padrões

In [ ]:
if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    dados, ibov = pd.read_pickle(CACHE)
else:
    dados = baixa(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, group_by="ticker")
    ibov = baixa("^BVSP", start="2005-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    pd.to_pickle((dados, ibov), CACHE)

CORRIGIDAS = {}                            # o que a limpeza mudou (registro da Parte 7; sem repetir)
SUSPEITOS = {"EQTL3.SA": ("2010-08-25", "queda de 40% sem volta (provável ajuste de bonificação ou desdobramento faltando)")}


def limpa(d, nome=""):
    """Limpeza do Capítulo 26, mais as três regras do Capítulo 28 para dado sujo que engana o teste de stops:
    1) máxima impossível: pavio de mais de 50% acima da abertura e do fechamento vira o maior dos dois;
    2) desdobramento ou grupamento sem ajuste: salto de mais de 2 vezes (ou menos da metade) da abertura em relação ao
       fechamento anterior, sem volta nos 5 pregões seguintes, corrige o histórico anterior pela mesma proporção;
    3) buraco nos dados: com mais de 15 dias corridos sem cotação, só fica o trecho contínuo depois do último buraco."""
    d = d.dropna().copy()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    d = d[ok & (d["Volume"] > 0)] if "Volume" in d else d[ok]
    px = ["Open", "High", "Low", "Close"]
    topo = d[["Open", "Close"]].max(axis=1)
    alta = d["High"] > 1.5 * topo
    for dt in d.index[alta]:
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: máxima {d.at[dt, 'High']:.2f} com fechamento {d.at[dt, 'Close']:.2f}", None)
    d.loc[alta, "High"] = topo[alta]
    salto = d["Open"] / d["Close"].shift()
    dura = d["Close"].shift(-5) / d["Close"].shift()
    for dt in d.index[((salto < 0.5) | (salto > 2)) & ((dura < 0.6) | (dura > 1.6))]:
        f_ = salto.loc[dt]
        d.loc[d.index < dt, px] *= f_
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: salto de {f_:.4f} vezes sem volta (ajuste que faltou); histórico"
                          " anterior corrigido", None)
    if nome == "SUZB3.SA":                 # antes disso, preço parado no Yahoo (Capítulo 26)
        d = d.loc["2017-11-10":]
    if nome in SUSPEITOS:                  # queda sem volta que parece ajuste faltando, mas sem prova: fica fora
        dt, motivo = SUSPEITOS[nome]
        CORRIGIDAS.setdefault(f"{nome} {pd.Timestamp(dt):%d/%m/%Y}: {motivo}; usa só a partir dessa data", None)
        d = d.loc[dt:]
    buraco = d.index.to_series().diff().dt.days
    if (buraco > 15).any():
        ini = buraco[buraco > 15].index[-1]
        CORRIGIDAS.setdefault(f"{nome}: {int((buraco > 15).sum())} buracos de mais de 15 dias; usa só a partir de {ini:%d/%m/%Y}", None)
        d = d.loc[ini:]
    return d


# Os padrões do Anexo I, com o nome da função da TA-Lib e o sentido esperado
# (+1 alta, -1 baixa, 0 = a TA-Lib dá o sentido pelo sinal: +100 alta, -100 baixa; None = doji, sem sentido)
PADROES = {
    "Bebê abandonado": ("CDLABANDONEDBABY", 0), "Nuvem negra": ("CDLDARKCLOUDCOVER", -1),
    "Doji": ("CDLDOJI", None), "Gap tasuki": ("CDLTASUKIGAP", 0), "Doji libélula": ("CDLDRAGONFLYDOJI", None),
    "Engolfo": ("CDLENGULFING", 0), "Estrela doji da noite": ("CDLEVENINGDOJISTAR", -1),
    "Estrela da noite": ("CDLEVENINGSTAR", -1), "Três métodos (alta e queda)": ("CDLRISEFALL3METHODS", 0),
    "Doji lápide": ("CDLGRAVESTONEDOJI", None), "Martelo": ("CDLHAMMER", 1), "Enforcado": ("CDLHANGINGMAN", -1),
    "Harami": ("CDLHARAMI", 0), "Harami cruz": ("CDLHARAMICROSS", 0), "Dia longo": ("CDLLONGLINE", 0),
    "Doji de pernas longas": ("CDLLONGLEGGEDDOJI", None), "Marubozu": ("CDLMARUBOZU", 0),
    "Estrela doji da manhã": ("CDLMORNINGDOJISTAR", 1), "Estrela da manhã": ("CDLMORNINGSTAR", 1),
    "Padrão perfurante": ("CDLPIERCING", 1), "Estrela cadente": ("CDLSHOOTINGSTAR", -1),
    "Dia curto": ("CDLSHORTLINE", 0), "Peão": ("CDLSPINNINGTOP", 0), "Vela prensada": ("CDLSTICKSANDWICH", 1),
    "Três corvos pretos": ("CDL3BLACKCROWS", -1), "Três soldados brancos": ("CDL3WHITESOLDIERS", 1),
    "Gap de alta com dois corvos": ("CDLUPSIDEGAP2CROWS", -1), "Martelo invertido": ("CDLINVERTEDHAMMER", 1),
}
REVERSAO = ["Martelo", "Martelo invertido", "Estrela da manhã", "Estrela doji da manhã", "Padrão perfurante", "Engolfo",
            "Harami", "Nuvem negra", "Estrela da noite", "Estrela doji da noite", "Estrela cadente", "Enforcado"]
DOJIS = ["Doji", "Doji libélula", "Doji lápide", "Doji de pernas longas"]

LIMPOS = {tk: limpa(dados[tk], tk) for tk in TICKERS}
linhas, todos5, todos_abs5 = [], [], []
for tk, d in LIMPOS.items():
    o, h, l, c = (d[k].to_numpy(dtype=float) for k in ("Open", "High", "Low", "Close"))
    cs = d["Close"]
    r5, r10 = cs.shift(-5) / cs - 1, cs.shift(-10) / cs - 1
    antes = cs / cs.shift(10) - 1                                   # tendência dos 10 pregões anteriores
    todos5.append(r5.dropna())
    todos_abs5.append(r5.dropna().abs())
    base5, base10 = r5.mean(), r10.mean()
    for nome, (fn, sentido) in PADROES.items():
        s = pd.Series(getattr(talib, fn)(o, h, l, c), index=d.index)
        for dt, v in s[s != 0].items():
            lado = 0 if sentido is None else (int(np.sign(v)) if sentido == 0 else sentido)
            linhas.append({"ação": tk, "padrão": nome, "data": dt, "lado": lado, "r5": r5.get(dt), "r10": r10.get(dt),
                           "base5": base5, "base10": base10, "antes": antes.get(dt), "abs5": abs(r5.get(dt, np.nan))})
EV = pd.DataFrame(linhas).dropna(subset=["r5", "r10", "antes"])
com_lado = EV["lado"] != 0
EV["ganho5"] = np.where(com_lado, EV["lado"] * (EV["r5"] - EV["base5"]), np.nan)
EV["ganho10"] = np.where(com_lado, EV["lado"] * (EV["r10"] - EV["base10"]), np.nan)
EV["acerto5"] = np.where(com_lado, (EV["lado"] * EV["r5"] > 0).astype(float), np.nan)
n_dias = sum(len(d) for d in LIMPOS.values())
pc = lambda x, k=2: "" if pd.isna(x) else f"{x:.{k}%}"

mostra(f"Parte 1. {len(TICKERS)} ações, {n_dias:_} pregões (2005 a 30/09/2026), limpeza do Capítulo 28")
for item in CORRIGIDAS:
    mostra("   corrigido:", item)
tab = EV.groupby("padrão").agg(n=("r5", "size"), ganho5=("ganho5", "mean"), ganho10=("ganho10", "mean"),
                               acerto5=("acerto5", "mean"))
tab["por mil"] = tab["n"] / n_dias * 1000
tab["t"] = tab["ganho5"] / (EV.groupby("padrão")["ganho5"].std() / np.sqrt(tab["n"]))   # eventos independentes: otimista
tab = tab.sort_values("por mil", ascending=False)
mostra("   padrão                          casos  por mil pregões")
for nome, lin in tab.iterrows():
    mostra(f"   {nome:28s} {int(lin['n']):>9_} {lin['por mil']:>10.1f}")

## Parte 2: os 5 e 10 pregões seguintes

In [ ]:
mostra("\nParte 2. Ganho = retorno depois do padrão menos o de um dia qualquer da mesma ação, com o sinal trocado nos"
       " padrões de baixa (positivo = a favor do padrão)")
mostra("   padrão                       ganho 5 d  ganho 10 d  acerto 5 d     t")
for nome, lin in tab[tab["ganho5"].notna()].iterrows():
    mostra(f"   {nome:28s} {pc(lin['ganho5']):>9s} {pc(lin['ganho10']):>10s} {pc(lin['acerto5'], 0):>10s} {lin['t']:+6.1f}")
mostra(f"   um dia qualquer: alta em 5 pregões em {pc((pd.concat(todos5) > 0).mean(), 0)} das vezes")
dj = EV[EV["padrão"].isin(DOJIS)]
mostra(f"   dojis: movimento absoluto médio em 5 pregões {pc(dj['abs5'].mean())} contra {pc(pd.concat(todos_abs5).mean())}"
       " de um dia qualquer")

In [ ]:
import matplotlib.pyplot as plt
g = tab[tab["ganho5"].notna()].sort_values("ganho5")
plt.figure(figsize=(8, 6))
plt.barh(g.index, g["ganho5"] * 100, color="gray")
plt.axvline(0, color="black")
plt.xlabel("ganho médio em 5 pregões (%) contra um dia qualquer")
plt.show()

## Parte 3: reversão no lugar certo

In [ ]:
mostra("\nParte 3. Reversão no lugar certo: alta depois de queda de 5% ou mais em 10 pregões; baixa depois de alta de 5%")
rv = EV[EV["padrão"].isin(REVERSAO) & com_lado].copy()
rv["no lugar"] = np.where(rv["lado"] > 0, rv["antes"] <= -LUGAR, rv["antes"] >= LUGAR)
t3 = rv.groupby(["padrão", "no lugar"]).agg(n=("ganho5", "size"), g=("ganho5", "mean"), a=("acerto5", "mean")).unstack()
mostra("   padrão                    fora: n   ganho  acerto | no lugar: n   ganho  acerto")
for nome, lin in t3.iterrows():
    mostra(f"   {nome:24s} {int(lin[('n', False)]):>8_} {pc(lin[('g', False)]):>7s} {pc(lin[('a', False)], 0):>6s} |"
           f" {int(lin[('n', True)]):>11_} {pc(lin[('g', True)]):>7s} {pc(lin[('a', True)], 0):>6s}")
tot = rv.groupby("no lugar")["ganho5"].agg(["size", "mean"])
mostra(f"   todos os de reversão: no lugar {pc(tot.loc[True, 'mean'])} (n = {int(tot.loc[True, 'size']):_})"
       f" | fora {pc(tot.loc[False, 'mean'])} (n = {int(tot.loc[False, 'size']):_})")

## Parte 4: estabilidade

In [ ]:
mostra("\nParte 4. Ganho em 5 pregões, 2005 a 2015 contra 2016 a 2026 (padrões com sentido e 300 ou mais casos em cada metade)")
EV["período"] = np.where(EV["data"] < CORTE, "2005-2015", "2016-2026")
t4 = EV[com_lado].groupby(["padrão", "período"])["ganho5"].agg(["size", "mean"]).unstack()
t4 = t4[(t4[("size", "2005-2015")] >= 300) & (t4[("size", "2016-2026")] >= 300)]
mesmo = np.sign(t4[("mean", "2005-2015")]) == np.sign(t4[("mean", "2016-2026")])
for nome, lin in t4.iterrows():
    mostra(f"   {nome:28s} {pc(lin[('mean', '2005-2015')]):>7s} | {pc(lin[('mean', '2016-2026')]):>7s}")
cor = np.corrcoef(t4[("mean", "2005-2015")], t4[("mean", "2016-2026")])[0, 1]
mostra(f"   mesmo sinal nas duas metades: {int(mesmo.sum())} de {len(t4)}; correlação entre as metades {cor:+.2f}")
METADES = t4                                   # usado na figura do anexo

## Parte 5: a definição muda o resultado

In [ ]:
def nison(d):
    """Engolfo e martelo pela regra de Nison, com a tendência anterior (queda de 5% em 10 pregões para os de alta).
    Engolfo de alta: corpo branco que envolve o corpo preto anterior (as sombras não contam).
    Martelo: sombra inferior de pelo menos 2 vezes o corpo, sombra superior de no máximo 10% da amplitude."""
    o, h, l, c = d["Open"], d["High"], d["Low"], d["Close"]
    corpo = (c - o).abs()
    queda = c.shift(1) / c.shift(11) - 1 <= -LUGAR            # tendência até o pregão anterior
    eng = (c.shift() < o.shift()) & (c > o) & (o <= c.shift()) & (c >= o.shift()) & (corpo > corpo.shift()) & queda
    inf = np.minimum(o, c) - l
    sup = h - np.maximum(o, c)
    mart = (inf >= 2 * corpo) & (sup <= 0.1 * (h - l)) & (corpo > 0) & queda
    return eng, mart


mostra("\nParte 5. A definição muda o resultado: os de alta pela TA-Lib (sem filtro) e pela regra de Nison (com a"
       " queda anterior)")
res = {k: [] for k in ("eng_ta", "eng_ni", "mar_ta", "mar_ni")}
sobre = {"eng": [0, 0, 0], "mar": [0, 0, 0]}
for tk, d in LIMPOS.items():
    o, h, l, c = (d[k].to_numpy(dtype=float) for k in ("Open", "High", "Low", "Close"))
    cs = d["Close"]
    g5 = cs.shift(-5) / cs - 1 - (cs.shift(-5) / cs - 1).mean()
    eng_ta = pd.Series(talib.CDLENGULFING(o, h, l, c), index=d.index) > 0
    mar_ta = pd.Series(talib.CDLHAMMER(o, h, l, c), index=d.index) > 0
    eng_ni, mar_ni = nison(d)
    for k, m in (("eng_ta", eng_ta), ("eng_ni", eng_ni), ("mar_ta", mar_ta), ("mar_ni", mar_ni)):
        res[k].append(g5[m].dropna())
    for k, a, b in (("eng", eng_ta, eng_ni), ("mar", mar_ta, mar_ni)):
        sobre[k][0] += int(a.sum())
        sobre[k][1] += int(b.sum())
        sobre[k][2] += int((a & b).sum())
for k, nome in (("eng", "Engolfo de alta"), ("mar", "Martelo")):
    ta, ni = pd.concat(res[k + "_ta"]), pd.concat(res[k + "_ni"])
    mostra(f"   {nome}: TA-Lib {sobre[k][0]:_} dias, ganho médio {pc(ta.mean())}, mediano {pc(ta.median())},"
           f" alta em {pc((ta > 0).mean(), 0)} | Nison {sobre[k][1]:_} dias, ganho médio {pc(ni.mean())},"
           f" mediano {pc(ni.median())}, alta em {pc((ni > 0).mean(), 0)}")
    mostra(f"      marcados pelos dois: {sobre[k][2]:_} ({sobre[k][2] / sobre[k][1]:.0%} dos de Nison)")
mostra("   (ganho = retorno de 5 pregões menos o de um dia qualquer da mesma ação; a média sofre com março de 2020)")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `nison` da Parte 5. Mostre por que a queda anterior usa `c.shift(1) / c.shift(11)` e como `inf >= 2 * corpo` traduz a sombra inferior de pelo menos duas vezes o corpo." Confira com a Parte 5, que imprime quantos martelos a regra de Nison e a TA-Lib marcaram juntas: 32% dos de Nison, como no livro.
2. **Adaptar:** "Numa célula nova depois da Parte 5, baixe [ATIVO] com a função `baixa`, com o ticker do Yahoo Finance, limpe os dados com a função `limpa` e rode `nison` e `talib.CDLHAMMER` só nele. Liste as datas marcadas pelas duas regras. Não mude a função `nison` nem a lista `TICKERS`." Antes de olhar o ativo novo, confira se a Parte 1 continua dando 3.971 martelos pela TA-Lib e a Parte 5, 32%, como no livro.

Mais pedidos deste anexo (entender e aplicar ao seu gráfico) estão na página do anexo, no hub do livro.

## Para ir além
1. Na Parte 3, troque o lugar certo de 5% para 10% em 10 pregões. O padrão perfurante continua bem?
2. Na Parte 2, troque os 5 pregões por 1 pregão. Os padrões de força (dia longo, marubozu) também devolvem o movimento no dia seguinte?
3. Na Parte 5, troque a regra do martelo para sombra de 3 vezes o corpo. Quantos martelos sobram?
4. Troque a lista de ações pelas que você opera e rode tudo de novo. O que mudou na Parte 4?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.